# broadband_mean_electrode
Get mean power per electrodes per stimulus.

Returns: NxM (N number of stimulus, M mean per electrode)

**Possible issues:**
1. Most electrodes are irrelevant to the decision, so we will have lots of noise. We will probably need to do an electrode selection before actually getting mean power.
2. Getting only the mean power across the time stimulus will probably be too little data. Perhaps, we should make more time windows inside a trial and also use those as features (considering we will end up with less electrodes after reduction),

In [3]:
# @title Data retrieval
import os, requests

fname = 'faceshouses.npz'
url = "https://osf.io/argh7/download"

if not os.path.isfile(fname):
  try:
    r = requests.get(url)
  except requests.ConnectionError:
    print("!!! Failed to download data !!!")
  else:
    if r.status_code != requests.codes.ok:
      print("!!! Failed to download data !!!")
    else:
      with open(fname, "wb") as fid:
        fid.write(r.content)

In [ ]:
# @title Imports

from matplotlib import rcParams
from matplotlib import pyplot as plt
rcParams['figure.figsize'] = [20, 4]
rcParams['font.size'] = 15
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['figure.autolayout'] = True

In [5]:
# @title Data loading
import numpy as np

alldat = np.load(fname, allow_pickle=True)['dat']

In [6]:
# @title Configure

subject_index = 1 #1 and 2 missing data on dat2

dat1 = alldat[subject_index][0]
dat2 = alldat[subject_index][1]

In [8]:
# @title Get broadband
from scipy import signal

V = dat1['V'].astype('float32')

# Remove slower
b, a = signal.butter(3, [50], btype='high', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# power
V = np.abs(V)**2

# Remove fast variations
b, a = signal.butter(3, [10], btype='low', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# Normalize to 0
V = V/V.mean(0)

In [28]:
'''
Slice voltages in stimulus time windows
'''

# Window preset per Kai Miller's paper
start_offset = -199
end_offset = 400

# Epoch
epochs_list = []

for on_time in dat1['t_on']:
    start = on_time + start_offset
    end = on_time + end_offset

    # Window
    trial_window = V[start:end]
    epochs_list.append(trial_window)

# Convert no np.array
epochs = np.array(epochs_list)
mean_epochs = np.mean(epochs, axis=1)

In [39]:
faces_trials = []
houses_trials = []

for idx, mean_epoch in enumerate(mean_epochs):
  if int(dat1['stim_id'][idx]) > 50:
    faces_trials.append(mean_epoch)
  else:
    houses_trials.append(mean_epoch)